<a href="https://colab.research.google.com/github/sandyreda898-oss/Smart-Agricultural-Water-Management/blob/main/weather_soil_water_modeling/notebooks/01_weather_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import os

repo_path = "/content/Smart-Agricultural-Water-Management"

if not os.path.exists(repo_path):
    !git clone https://github.com/sandyreda898-oss/Smart-Agricultural-Water-Management.git
else:
    print("Repository already exists. Using the current copy.")

Repository already exists. Using the current copy.


In [38]:
import pandas as pd
import numpy as np

base = "/content/Smart-Agricultural-Water-Management/AgriWater/AgriWater_Clean_Data/data/table_exports/"

weather = pd.read_csv(base + "weather_daily.csv.gz")
grid = pd.read_csv(base + "grid_point.csv")

In [3]:
print("Weather shape:", weather.shape)
print("Grid shape:", grid.shape)

Weather shape: (1089840, 6)
Grid shape: (38, 5)


In [4]:
weather.head()

,point_id,obs_date,parameter,value,unit,source_id
0,grid_0001,2017-01-01,ALLSKY_SFC_SW_DWN,11.32,MJ/m^2/day,1
1,grid_0001,2017-01-01,PRECTOTCORR,0.04,mm/day,1
2,grid_0001,2017-01-01,PS,102.26,kPa,1
3,grid_0001,2017-01-01,RH2M,64.90,%,1
4,grid_0001,2017-01-01,T2M,9.79,C,1


In [5]:
weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1089840 entries, 0 to 1089839
Data columns (total 6 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   point_id   1089840 non-null  object 
 1   obs_date   1089840 non-null  object 
 2   parameter  1089840 non-null  object 
 3   value      1088548 non-null  float64
 4   unit       1089840 non-null  object 
 5   source_id  1089840 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 49.9+ MB


In [6]:
grid.head()

,point_id,latitude,longitude,district_id,spatial_support
0,grid_0001,31.012854,30.818041,9,gridded estimate
1,grid_0002,31.012854,30.918041,9,gridded estimate
2,grid_0003,31.012854,31.018041,7,gridded estimate
3,grid_0004,31.112854,30.718041,5,gridded estimate
4,grid_0005,31.112854,30.818041,9,gridded estimate


In [7]:
weather.isnull().sum()

,0
point_id,0
obs_date,0
parameter,0
value,1292
unit,0
source_id,0


In [8]:
print("Duplicate rows:", weather.duplicated().sum())

Duplicate rows: 0


In [9]:
key_duplicates = weather.duplicated(
    subset=["point_id", "obs_date", "parameter"]).sum()

print("Duplicate point-date-parameter rows:", key_duplicates)

Duplicate point-date-parameter rows: 0


In [10]:
weather["obs_date"] = pd.to_datetime(
    weather["obs_date"],
    errors="coerce")

print("First date:", weather["obs_date"].min())
print("Last date:", weather["obs_date"].max())

First date: 2017-01-01 00:00:00
Last date: 2026-10-25 00:00:00


In [11]:
print("Number of weather parameters:", weather["parameter"].nunique())

weather["parameter"].value_counts()

Number of weather parameters: 8


,count
parameter,
ALLSKY_SFC_SW_DWN,136230
PRECTOTCORR,136230
PS,136230
RH2M,136230
T2M,136230
T2M_MAX,136230
T2M_MIN,136230
WS2M,136230


In [12]:
weather.groupby("parameter")["unit"].unique()

,unit
parameter,
ALLSKY_SFC_SW_DWN,[MJ/m^2/day]
PRECTOTCORR,[mm/day]
PS,[kPa]
RH2M,[%]
T2M,[C]
T2M_MAX,[C]
T2M_MIN,[C]
WS2M,[m/s]


In [13]:
parameter_summary = (
    weather.groupby(["parameter", "unit"])
    .agg(
        total_records=("value", "size"),
        missing_values=("value", lambda x: x.isna().sum()),
        minimum=("value", "min"),
        maximum=("value", "max"),
        average=("value", "mean")).reset_index())

parameter_summary

,parameter,unit,total_records,missing_values,minimum,maximum,average
0,ALLSKY_SFC_SW_DWN,MJ/m^2/day,136230,228,2.07,30.35,19.641948
1,PRECTOTCORR,mm/day,136230,152,0.00,485.52,0.722831
2,PS,kPa,136230,152,99.78,103.02,101.364686
3,RH2M,%,136230,152,20.47,90.73,63.379540
4,T2M,C,136230,152,8.21,34.80,22.190861
5,T2M_MAX,C,136230,152,11.23,46.95,27.380426
6,T2M_MIN,C,136230,152,1.94,28.03,18.094419
7,WS2M,m/s,136230,152,0.75,10.60,3.569218


In [14]:
missing_weather = weather[
    weather["value"].isna()
].copy()

print("Total missing values:", len(missing_weather))

missing_weather.head()

Total missing values: 1292


,point_id,obs_date,parameter,value,unit,source_id
28632,grid_0001,2026-10-20,ALLSKY_SFC_SW_DWN,NaN,MJ/m^2/day,1
28640,grid_0001,2026-10-21,ALLSKY_SFC_SW_DWN,NaN,MJ/m^2/day,1
28648,grid_0001,2026-10-22,ALLSKY_SFC_SW_DWN,NaN,MJ/m^2/day,1
28649,grid_0001,2026-10-22,PRECTOTCORR,NaN,mm/day,1
28650,grid_0001,2026-10-22,PS,NaN,kPa,1


In [15]:
missing_weather["parameter"].value_counts()

,count
parameter,
ALLSKY_SFC_SW_DWN,228
PRECTOTCORR,152
PS,152
RH2M,152
T2M,152
T2M_MAX,152
T2M_MIN,152
WS2M,152


In [16]:
missing_weather.groupby("obs_date").size().sort_index()

,0
obs_date,
2026-10-20,38
2026-10-21,38
2026-10-22,304
2026-10-23,304
2026-10-24,304
2026-10-25,304


In [17]:
missing_weather["point_id"].value_counts().sort_index()

,count
point_id,
grid_0001,34
grid_0002,34
grid_0003,34
grid_0004,34
grid_0005,34
grid_0006,34
grid_0007,34
grid_0008,34
grid_0009,34


In [18]:
weather_wide = weather.pivot(
    index=["point_id", "obs_date"],
    columns="parameter",
    values="value").reset_index()

weather_wide.head()

parameter,point_id,obs_date,ALLSKY_SFC_SW_DWN,PRECTOTCORR,PS,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M
0,grid_0001,2017-01-01,11.32,0.04,102.26,64.90,9.79,16.73,5.38,2.26
1,grid_0001,2017-01-02,10.49,0.22,102.33,73.94,10.76,16.53,5.64,2.28
2,grid_0001,2017-01-03,11.33,0.15,102.42,74.23,11.64,17.38,6.94,1.60
3,grid_0001,2017-01-04,10.52,0.39,102.29,80.24,12.06,17.60,7.99,2.21
4,grid_0001,2017-01-05,10.12,0.00,102.14,63.24,11.77,17.80,5.71,1.33


In [19]:
print("Original shape:", weather.shape)
print("Wide shape:", weather_wide.shape)

Original shape: (1089840, 6)
Wide shape: (136230, 10)


In [20]:
grid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   point_id         38 non-null     object 
 1   latitude         38 non-null     float64
 2   longitude        38 non-null     float64
 3   district_id      38 non-null     int64  
 4   spatial_support  38 non-null     object 
dtypes: float64(2), int64(1), object(2)
memory usage: 1.6+ KB


In [21]:
print("Duplicate point IDs:", grid["point_id"].duplicated().sum())
print("Number of grid points:", grid["point_id"].nunique())

Duplicate point IDs: 0
Number of grid points: 38


In [22]:
weather_wide = weather_wide.merge(
    grid,
    on="point_id",
    how="left",
    validate="many_to_one")

weather_wide.head()

,point_id,obs_date,ALLSKY_SFC_SW_DWN,PRECTOTCORR,PS,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M,latitude,longitude,district_id,spatial_support
0,grid_0001,2017-01-01,11.32,0.04,102.26,64.90,9.79,16.73,5.38,2.26,31.012854,30.818041,9,gridded estimate
1,grid_0001,2017-01-02,10.49,0.22,102.33,73.94,10.76,16.53,5.64,2.28,31.012854,30.818041,9,gridded estimate
2,grid_0001,2017-01-03,11.33,0.15,102.42,74.23,11.64,17.38,6.94,1.60,31.012854,30.818041,9,gridded estimate
3,grid_0001,2017-01-04,10.52,0.39,102.29,80.24,12.06,17.60,7.99,2.21,31.012854,30.818041,9,gridded estimate
4,grid_0001,2017-01-05,10.12,0.00,102.14,63.24,11.77,17.80,5.71,1.33,31.012854,30.818041,9,gridded estimate


In [23]:
weather_wide.isnull().sum()

,0
point_id,0
obs_date,0
ALLSKY_SFC_SW_DWN,228
PRECTOTCORR,152
PS,152
RH2M,152
T2M,152
T2M_MAX,152
T2M_MIN,152
WS2M,152


In [24]:
et0_inputs = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS2M",
    "PS",
    "ALLSKY_SFC_SW_DWN"]

weather_wide["et0_inputs_complete"] = (
    weather_wide[et0_inputs]
    .notna()
    .all(axis=1))

weather_wide["et0_inputs_complete"].value_counts()

,count
et0_inputs_complete,
True,136002
False,228


In [25]:
weather_wide["rainfall_available"] = (
    weather_wide["PRECTOTCORR"].notna())

weather_wide["rainfall_available"].value_counts()

,count
rainfall_available,
True,136078
False,152


In [26]:
weather_wide[
    weather_wide["latitude"].isna()
][["point_id"]].drop_duplicates()

,point_id


In [27]:
date_coverage = (
    weather_wide.groupby("point_id")
    .agg(
        first_date=("obs_date", "min"),
        last_date=("obs_date", "max"),
        number_of_days=("obs_date", "nunique")))

date_coverage

,first_date,last_date,number_of_days
point_id,,,
grid_0001,2017-01-01,2026-10-25,3585
grid_0002,2017-01-01,2026-10-25,3585
grid_0003,2017-01-01,2026-10-25,3585
grid_0004,2017-01-01,2026-10-25,3585
grid_0005,2017-01-01,2026-10-25,3585
grid_0006,2017-01-01,2026-10-25,3585
grid_0007,2017-01-01,2026-10-25,3585
grid_0008,2017-01-01,2026-10-25,3585
grid_0009,2017-01-01,2026-10-25,3585


In [28]:
weather_wide.head(10)

,point_id,obs_date,ALLSKY_SFC_SW_DWN,PRECTOTCORR,PS,RH2M,T2M,T2M_MAX,T2M_MIN,WS2M,latitude,longitude,district_id,spatial_support,et0_inputs_complete,rainfall_available
0,grid_0001,2017-01-01,11.32,0.04,102.26,64.90,9.79,16.73,5.38,2.26,31.012854,30.818041,9,gridded estimate,True,True
1,grid_0001,2017-01-02,10.49,0.22,102.33,73.94,10.76,16.53,5.64,2.28,31.012854,30.818041,9,gridded estimate,True,True
2,grid_0001,2017-01-03,11.33,0.15,102.42,74.23,11.64,17.38,6.94,1.60,31.012854,30.818041,9,gridded estimate,True,True
3,grid_0001,2017-01-04,10.52,0.39,102.29,80.24,12.06,17.60,7.99,2.21,31.012854,30.818041,9,gridded estimate,True,True
4,grid_0001,2017-01-05,10.12,0.00,102.14,63.24,11.77,17.80,5.71,1.33,31.012854,30.818041,9,gridded estimate,True,True
5,grid_0001,2017-01-06,12.60,0.00,101.54,60.81,11.77,19.78,5.73,1.19,31.012854,30.818041,9,gridded estimate,True,True
6,grid_0001,2017-01-07,10.82,0.00,101.50,60.71,9.62,16.04,4.01,4.58,31.012854,30.818041,9,gridded estimate,True,True
7,grid_0001,2017-01-08,8.11,0.01,101.38,60.68,9.57,14.76,5.62,5.92,31.012854,30.818041,9,gridded estimate,True,True
8,grid_0001,2017-01-09,8.81,0.02,101.58,59.53,8.57,14.95,3.91,4.08,31.012854,30.818041,9,gridded estimate,True,True
9,grid_0001,2017-01-10,12.35,0.00,101.68,59.98,8.67,16.74,2.76,3.05,31.012854,30.818041,9,gridded estimate,True,True


In [29]:
weather_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136230 entries, 0 to 136229
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   point_id             136230 non-null  object        
 1   obs_date             136230 non-null  datetime64[ns]
 2   ALLSKY_SFC_SW_DWN    136002 non-null  float64       
 3   PRECTOTCORR          136078 non-null  float64       
 4   PS                   136078 non-null  float64       
 5   RH2M                 136078 non-null  float64       
 6   T2M                  136078 non-null  float64       
 7   T2M_MAX              136078 non-null  float64       
 8   T2M_MIN              136078 non-null  float64       
 9   WS2M                 136078 non-null  float64       
 10  latitude             136230 non-null  float64       
 11  longitude            136230 non-null  float64       
 12  district_id          136230 non-null  int64         
 13  spatial_suppor

In [46]:
output_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/weather_daily_prepared.csv.gz"
)

weather_wide.to_csv(
    output_path,
    index=False,
    compression="gzip"
)

print("Prepared weather data saved successfully.")
print(output_path)

Prepared weather data saved successfully.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/weather_daily_prepared.csv.gz


In [32]:
incomplete_rows = weather_wide[
    (~weather_wide["et0_inputs_complete"]) |
    (~weather_wide["rainfall_available"])
]

print("Incomplete rows:", len(incomplete_rows))

incomplete_rows[
    ["point_id", "obs_date", "et0_inputs_complete", "rainfall_available"]
].tail(20)

Incomplete rows: 228


,point_id,obs_date,et0_inputs_complete,rainfall_available
125473,grid_0035,2026-10-24,False,False
125474,grid_0035,2026-10-25,False,False
129054,grid_0036,2026-10-20,False,True
129055,grid_0036,2026-10-21,False,True
129056,grid_0036,2026-10-22,False,False
129057,grid_0036,2026-10-23,False,False
129058,grid_0036,2026-10-24,False,False
129059,grid_0036,2026-10-25,False,False
132639,grid_0037,2026-10-20,False,True
132640,grid_0037,2026-10-21,False,True


In [33]:
weather_clean = weather_wide[
    (weather_wide["et0_inputs_complete"]) &
    (weather_wide["rainfall_available"])
].copy()

print("Prepared rows:", len(weather_wide))
print("Clean modeling rows:", len(weather_clean))
print("Rows excluded:", len(weather_wide) - len(weather_clean))

Prepared rows: 136230
Clean modeling rows: 136002
Rows excluded: 228


In [34]:
weather_clean[
    [
        "T2M",
        "T2M_MAX",
        "T2M_MIN",
        "RH2M",
        "PRECTOTCORR",
        "WS2M",
        "PS",
        "ALLSKY_SFC_SW_DWN"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
T2M,136002.0,22.191006,5.570722,8.21,17.20,22.58,27.34,34.80
T2M_MAX,136002.0,27.380797,6.824864,11.23,21.56,27.49,32.04,46.95
T2M_MIN,136002.0,18.094328,5.340226,1.94,13.81,18.39,22.78,28.03
RH2M,136002.0,63.379274,8.529603,20.47,57.93,64.45,69.18,90.73
PRECTOTCORR,136002.0,0.722174,8.018197,0.00,0.00,0.00,0.03,485.52
WS2M,136002.0,3.569608,1.312085,0.75,2.66,3.36,4.31,10.60
PS,136002.0,101.364290,0.490079,99.78,100.99,101.35,101.69,103.02
ALLSKY_SFC_SW_DWN,136002.0,19.641948,6.749693,2.07,13.30,20.41,25.97,30.35


In [47]:
clean_output_path = (
    "/content/Smart-Agricultural-Water-Management/"
    "weather_soil_water_modeling/"
    "data/processed/weather_daily_clean.csv.gz"
)

weather_clean.to_csv(
    clean_output_path,
    index=False,
    compression="gzip"
)

print("Clean weather data saved successfully.")
print(clean_output_path)

Clean weather data saved successfully.
/content/Smart-Agricultural-Water-Management/weather_soil_water_modeling/data/processed/weather_daily_clean.csv.gz


In [48]:
import os

print("Prepared exists:", os.path.exists(output_path))
print("Clean exists:", os.path.exists(clean_output_path))

Prepared exists: True
Clean exists: True
